# 12장 Fine-Tuning Generation Models — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter12/Chapter 12 - Fine-tuning Generation Models.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter12/Chapter%2012%20-%20Fine-tuning%20Generation%20Models.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter12` 노트북(Apache 2.0)을 현재 라이브러리 버전에 맞게 수정. 사전학습만 마친 **베이스 모델(TinyLlama 1.1B)**을 두 단계로 다듬습니다. 먼저 지시를 따르게 만드는 **지도 미세조정(SFT, 여기서는 instruction tuning)**을 하고, 이어서 사람이 선호하는 답 쪽으로 기울이는 **선호 튜닝(preference tuning, 여기서는 DPO)**을 합니다. 둘 다 4비트 양자화 + LoRA, 즉 **QLoRA**로 진행합니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | LLM 학습 3단계, 전체 미세조정 vs PEFT(어댑터·LoRA·양자화) 개념 | 487–502 |
| 2 | 지시 데이터를 채팅 템플릿으로 바꾸기 | 503–505 |
| 3 | 베이스 모델을 4비트로 불러오기(QLoRA의 Q) | 505–506 |
| 4 | LoRA 설정 | 506–508 |
| 5 | SFT 학습 | 508–510 |
| 6 | 어댑터 병합과 추론 | 510–511 |
| 7 | 생성 모델 평가 방법 (+ 퍼플렉서티 측정) | 511–518 |
| 8 | 선호 튜닝·보상 모델(PPO) 개념 | 518–527 |
| 9 | DPO 학습과 병합 | 528–533 |
| 10 | (+ 보충) 베이스 → SFT → DPO 답변 비교, 선호 마진 측정 | — |

**권장 환경:** `ml.g4dn.xlarge`(T4 16GB)로 책과 같은 조건에서 돌아갑니다(SFT 약 13분, DPO 약 13분, 전체 약 40~50분). 쿼터가 있다면 `ml.g5.xlarge`(A10G 24GB)가 더 빠르고 bf16도 지원하지만, 이 노트북은 T4에 맞춰 fp16으로 둡니다. 스토리지는 30GB 이상(TinyLlama 원본 가중치 4.4GB, 어댑터 2개, 데이터셋 포함 약 6GB).

**빠른 실행:** `QUICK = True`면 SFT 300개, DPO 20스텝으로 줄여 흐름만 15분 안팎에 확인합니다. 학습이 덜 되어 답변 변화는 작게 나옵니다.

`(+ …)` 표시는 책에 없는 보충 실험입니다.

## 0. 환경 점검과 패키지 설치

책은 trl 0.9.4 / peft 0.11.1 / transformers 4.41.2에 고정되어 있습니다. 그 뒤 TRL의 인자 체계가 크게 바뀌어(학습 옵션을 `SFTConfig`·`DPOConfig`로 이동, `tokenizer=` → `processing_class=` 등) 책 코드는 최신 버전에서 그대로 돌지 않습니다. 옛 버전에 맞추면 최신 Studio 이미지의 torch·CUDA와 충돌할 수 있어, **현재 버전에 맞게 코드를 고치고 검증한 조합으로 고정**했습니다. 설치가 일어나면 **커널을 재시작**한 뒤 이 셀부터 다시 실행하세요.

> bitsandbytes 4비트는 NVIDIA GPU에서 SM60 이상을 지원하고 SM75 이상을 권장합니다(공식 README). T4가 정확히 SM75입니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!df -h ~ | tail -1
import importlib.metadata as md, subprocess, sys
PINS = {"transformers": "5.20.2", "trl": "1.14.2", "peft": "0.21.2", "datasets": "5.2.0",
        "accelerate": "1.15.0", "bitsandbytes": "0.50.2"}
def ver(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
todo = [f"{p}=={v}" for p, v in PINS.items() if ver(p) != v]
if todo:
    print("설치:", todo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
    print("\n설치 완료 → 메뉴 Kernel > Restart Kernel 후 이 셀부터 다시 실행하세요.")
else:
    print("검증된 버전 그대로입니다.")

In [ ]:
import gc, math, torch, pandas as pd, transformers, trl, peft
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)

QUICK = False                       # True: SFT 300개, DPO 20스텝
N_SFT = 300 if QUICK else 3_000     # 책: 3,000
DPO_STEPS = 20 if QUICK else 200    # 책: 200
answers = {}                        # 같은 질문에 대한 단계별 답변을 모아 10절에서 비교

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 1. LLM 학습 3단계와 PEFT (PDF 487–502)

1. **사전학습(pretraining):** 대량 텍스트로 다음 토큰 예측. 결과는 지시를 따르지 않는 베이스 모델입니다.
2. **지도 미세조정(SFT):** 질문-답변 형식의 지시 데이터로 학습해 지시를 따르게 만듭니다.
3. **선호 튜닝(preference tuning):** 같은 질문의 두 답 중 사람이 고른 쪽을 더 잘 내도록 정렬(alignment)합니다.

전체 미세조정(full fine-tuning)은 모든 가중치를 바꾸므로 메모리와 저장 공간이 많이 듭니다. **PEFT(parameter-efficient fine-tuning)**는 원래 가중치를 얼리고 작은 부분만 학습합니다.

- **어댑터(adapter):** Transformer 블록 사이에 작은 층을 끼워 넣고 그것만 학습합니다.
- **LoRA(low-rank adaptation):** 큰 가중치 행렬 W(d×k)를 직접 바꾸는 대신, 변화량을 작은 두 행렬의 곱 B·A(d×r, r×k)로 학습합니다. r(rank)이 작을수록 학습할 숫자가 줄어듭니다.
- **양자화(quantization):** 얼려 둔 원래 가중치를 4비트로 저장해 메모리를 줄입니다. 양자화 + LoRA = **QLoRA**입니다.

**(+ 보충) LoRA가 얼마나 줄이나.** TinyLlama의 어텐션 투영 행렬 하나(2048×2048)를 r=64로 바꿀 때를 계산합니다.

In [ ]:
d, k, r = 2048, 2048, 64
full, lora = d * k, r * (d + k)
print(f"전체 학습: {full:,}개 / LoRA(r={r}): {lora:,}개 → {lora/full:.1%}")
for r_ in [4, 8, 16, 32, 64]:
    print(f"  r={r_:>2}: {r_*(d+k):>9,}개 ({r_*(d+k)/full:.2%})")

## 2. 지시 데이터 템플릿 만들기 (PDF 503–505)

모델이 "사용자 말"과 "자기 답"을 구분하려면 일정한 채팅 템플릿이 필요합니다(그림 12-19). TinyLlama 채팅 버전이 쓰는 `<|user|> … </s> <|assistant|> …` 형식을 그대로 씁니다. 데이터는 UltraChat 20만 대화를 정제한 `ultrachat_200k`의 일부입니다.

> **책과 다른 점.** ① 책 코드는 `split="test_sft"`만 쓰지만 데이터셋의 **모든 분할(약 1.6GB)**을 내려받습니다. `data_files`로 필요한 파일(수십 MB)만 받게 했습니다. 고른 3,000개는 책과 같습니다. ② `remove_columns`를 추가해 `text` 열만 남깁니다. 원본에 `prompt` 열이 있어서, 최신 TRL은 이를 "프롬프트-응답" 형식 데이터로 판단하고 `completion` 열을 찾다가 오류를 냅니다.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """TinyLlama가 쓰는 <|user|> 템플릿으로 대화를 문자열 하나로 만든다"""
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

ultrachat = load_dataset("HuggingFaceH4/ultrachat_200k",
                         data_files={"test_sft": "data/test_sft-*.parquet"}, split="test_sft")
dataset = ultrachat.shuffle(seed=42).select(range(N_SFT))
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)
dataset

In [ ]:
# 책과 같은 예시(2576번). QUICK이면 범위 밖이라 0번을 보여 줌
print(dataset["text"][2576 if len(dataset) > 2576 else 0])

## 3. 4비트로 불러오기 — QLoRA의 Q (PDF 505–506)

`nf4`(정규분포에 맞춘 4비트 표현)로 저장하고, 실제 계산은 `float16`으로 합니다. 이중 양자화(double quantization)는 양자화에 쓰는 상수까지 한 번 더 압축합니다. 책 기준 약 4GB → 약 1GB입니다.

> **T4와 dtype.** T4는 bf16 연산을 하드웨어로 지원하지 않습니다. 책이 `bnb_4bit_compute_dtype="float16"`, 학습 옵션 `fp16=True`를 쓴 것도 이 때문입니다. 이 노트북도 그대로 둡니다.

> **책과 다른 점(pad 토큰).** 책은 `tokenizer.pad_token = "<PAD>"`로 정합니다. 그런데 `<PAD>`는 TinyLlama 어휘집에 없는 문자열이라, 예전 버전은 조용히 `<unk>`(ID 0)로 바꿔 썼고 최신 TRL은 "어휘집에 없는 pad 토큰"이라며 멈춥니다. 같은 ID 0인 `<unk>`를 직접 지정해 동작은 책과 같게 했습니다.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                   # 4비트로 불러오기
    bnb_4bit_quant_type="nf4",           # 양자화 형식
    bnb_4bit_compute_dtype="float16",    # 계산 dtype (T4: fp16)
    bnb_4bit_use_double_quant=True,      # 이중 양자화
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,      # 일반 SFT라면 이 줄을 뺀다
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = "<unk>"            # 책: "<PAD>" (어휘집에 없음 → 실제로는 ID 0 = <unk>)
tokenizer.padding_side = "left"
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)

**(+ 보충) 메모리 확인과 학습 전 답변.** 4비트 모델이 실제로 차지하는 크기를 보고, 학습 **전** 베이스 모델에 6절과 같은 질문을 던져 둡니다. 베이스 모델은 지시를 따르도록 학습되지 않았으므로 질문에 답하기보다 문장을 이어 쓰는 경향이 보일 것입니다.

In [ ]:
print(f"모델 메모리: {model.get_memory_footprint()/1e9:.2f} GB")

PROMPT = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
def ask(m, prompt=PROMPT, max_new_tokens=200):
    enc = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

answers["0. base (4-bit)"] = ask(model)
print(answers["0. base (4-bit)"])

## 4. LoRA 설정 (PDF 506–508)

- `r=64`: 저차원 행렬의 rank. 클수록 표현력이 늘고 압축은 줄어듭니다(보통 4~64).
- `lora_alpha=32`: 원래 가중치에 더하는 변화량의 크기 조절. 경험칙은 r의 2배입니다.
- `target_modules`: LoRA를 붙일 층. 어텐션(q/k/v/o)과 MLP(gate/up/down) 투영을 모두 지정했습니다.

`prepare_model_for_kbit_training`은 4비트 모델을 학습 가능한 상태로 준비합니다(일부 층을 fp32로 올리고 gradient checkpointing 활성화).

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj'],
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()      # (+ 보충) 실제로 학습하는 비율

## 5. SFT 학습 (PDF 508–510)

> **책과 다른 점.** ① 책은 `TrainingArguments`를 만들고 `SFTTrainer`에 `dataset_text_field`, `max_seq_length`, `tokenizer`를 직접 넘깁니다. 최신 TRL은 이 옵션들을 `SFTConfig` 안으로 옮겼고, `max_seq_length`는 `max_length`로, `tokenizer`는 `processing_class`로 이름이 바뀌었습니다. 값은 책과 같습니다. ② 책은 이미 `get_peft_model`로 감싼 모델과 `peft_config`를 **둘 다** 넘깁니다. 최신 TRL은 이 조합을 오류로 막으므로 `peft_config`는 넘기지 않습니다(4절에서 이미 적용됨). ③ `bf16=False`를 명시했습니다. T4에서 fp16으로 학습한다는 뜻을 분명히 하려는 것입니다.

T4에서 375스텝(3,000개 ÷ 배치 2 ÷ 누적 4) 약 13분 걸립니다. 손실은 10스텝마다 출력됩니다.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_arguments = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",      # QLoRA 논문의 paged 옵티마이저
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    dataset_text_field="text",      # 책: SFTTrainer 인자
    max_length=512,                 # 책: max_seq_length=512
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,     # 책: tokenizer=tokenizer
    args=training_arguments,
)
trainer.train()
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")   # LoRA 가중치(어댑터)만 저장

In [ ]:
# (+ 보충) 손실 곡선
logs = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])
if len(logs):
    ax = logs.plot(x="step", y="loss", marker="o", figsize=(7, 3), title="SFT training loss", legend=False)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.grid(alpha=.3)
else:
    print("logging_steps보다 학습 스텝이 적어 기록된 손실이 없습니다.")

In [ ]:
import os
print(sorted(os.listdir("TinyLlama-1.1B-qlora")))
print(f"어댑터 크기: {os.path.getsize('TinyLlama-1.1B-qlora/adapter_model.safetensors')/1e6:.0f} MB (원본 모델 약 4,400 MB)")
del model, trainer; free_gpu()

## 6. 어댑터 병합과 추론 (PDF 510–511)

학습한 LoRA 가중치를 원래 가중치에 더해 모델 하나로 만듭니다. 책 설명대로 이번에는 4비트가 아니라 **16비트로** 다시 불러와 병합합니다(4비트에 병합하면 반올림 오차가 생깁니다).

> 책 코드는 dtype을 지정하지 않습니다. 버전에 따라 32비트로 불러올 수 있어 `dtype=torch.float16`을 명시했습니다.

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
    dtype=torch.float16,           # 추가: 16비트로 병합
)
merged_model = model.merge_and_unload()

In [ ]:
from transformers import pipeline

prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

In [ ]:
answers["1. SFT"] = ask(merged_model)     # 10절 비교용(같은 생성 설정)

## 7. 생성 모델 평가 (PDF 511–518)

생성 모델은 정답이 하나로 정해지지 않아 평가가 어렵습니다. 책은 네 가지를 소개합니다.

| 방법 | 내용 | 한계 |
|---|---|---|
| 단어 수준 지표 | 퍼플렉서티, ROUGE, BLEU, BERTScore | 유창성·정확성·창의성을 반영하지 못함 |
| 벤치마크·리더보드 | MMLU, GSM8k, HellaSwag, HumanEval / Open LLM Leaderboard | 벤치마크 과적합, 특정 용도를 대표하지 못함 |
| 자동 평가 | LLM-as-a-judge: 다른 LLM이 답의 질을 채점, 두 답을 비교 | 심판 모델의 편향 |
| 사람 평가 | Chatbot Arena: 익명 두 모델 중 선호 투표 → Elo 점수 | 비용, 다수 의견이 내 용도와 다를 수 있음 |

책의 결론은 "의도한 용도로 직접 평가하라"입니다. 책은 Goodhart의 법칙("측정치가 목표가 되면 좋은 측정치가 아니게 된다")도 함께 인용합니다.

**(+ 보충) 퍼플렉서티 직접 재기.** 학습에 쓰지 않은 UltraChat 대화 30개에 대해 베이스 모델과 SFT 모델의 퍼플렉서티를 비교합니다. 퍼플렉서티 = exp(토큰당 평균 음의 로그우도)이고, 낮을수록 그 텍스트를 잘 예측한다는 뜻입니다. SFT 모델은 바로 이 형식(채팅 템플릿)으로 학습했으므로 낮아지는 것이 자연스럽습니다. 동시에 이 숫자가 "답이 맞는지"는 알려 주지 않는다는 점이 책이 말한 한계입니다.

In [ ]:
held_out = ultrachat.shuffle(seed=42).select(range(N_SFT, N_SFT + 30))
held_out = held_out.map(format_prompt, remove_columns=held_out.column_names)["text"]

def perplexity(m, texts, max_len=512):
    m.eval(); nll, n = 0.0, 0
    for t in texts:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_len).to(m.device)
        with torch.no_grad():
            loss = m(**enc, labels=enc["input_ids"]).loss.item()
        k = enc["input_ids"].shape[1] - 1
        nll += loss * k; n += k
    return math.exp(nll / n)

base16 = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
ppl = {"base": perplexity(base16, held_out), "SFT": perplexity(merged_model, held_out)}
del base16; free_gpu()
pd.Series(ppl, name="perplexity (held-out UltraChat)").round(2).to_frame()

In [ ]:
del model, merged_model, pipe; free_gpu()

## 8. 선호 튜닝과 보상 모델 (PDF 518–527)

SFT 모델은 지시를 따르지만, 같은 질문에도 더 좋은 답과 덜 좋은 답이 있습니다. 선호 튜닝은 그 차이를 학습합니다.

**보상 모델(reward model) 방식(RLHF).** ① 같은 프롬프트에 대한 두 답 중 사람이 고른 쪽(chosen)과 버린 쪽(rejected)을 모읍니다. ② SFT 모델 사본의 생성 헤드를 **점수 하나를 내는 헤드**로 바꿔, chosen 점수가 rejected보다 높아지도록 학습합니다. 이것이 보상 모델입니다. ③ 보상 모델을 채점자로 삼아 LLM을 강화학습(PPO, Proximal Policy Optimization)으로 학습합니다. ChatGPT 초기 버전이 이 방식이었습니다. 모델을 둘 이상 학습해야 해서 복잡하고 불안정할 수 있습니다.

**DPO(Direct Preference Optimization).** 보상 모델을 따로 두지 않습니다. 학습 중인 모델과 고정된 **참조 모델(reference model)**이 chosen·rejected 답에 매기는 로그 확률을 비교합니다. 그리고 참조 모델 대비 chosen의 확률은 올리고 rejected의 확률은 내리도록 직접 최적화합니다. 강화학습 루프가 없어 안정적이라 이 장의 주 방법으로 씁니다. 책은 SFT와 DPO를 한 번에 하는 ORPO도 소개합니다.

## 9. DPO 학습 (PDF 528–533)

선호 데이터는 `argilla/distilabel-intel-orca-dpo-pairs`입니다. 동점(tie)을 빼고, chosen 점수 8 이상, GSM8k 학습셋과 겹치지 않는 것만 남겨 약 13,000개에서 약 6,000개로 줄입니다.

> **책 본문과 코드의 차이.** 본문(528쪽)은 "전체 미세조정 후 DPO까지 거친 TinyLlama 지시 모델을 쓴다"고 설명하지만, 실제 코드는 5절에서 만든 **우리 SFT 어댑터**(`TinyLlama-1.1B-qlora`)를 불러옵니다. 이 노트북은 코드를 따릅니다.

In [ ]:
def format_prompt(example):
    """TinyLlama 템플릿으로 prompt / chosen / rejected를 만든다"""
    system = "<|system|>\n" + example['system'] + "</s>\n"
    prompt = "<|user|>\n" + example['input'] + "</s>\n<|assistant|>\n"
    chosen = example['chosen'] + "</s>\n"
    rejected = example['rejected'] + "</s>\n"
    return {"prompt": system + prompt, "chosen": chosen, "rejected": rejected}

dpo_dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
dpo_dataset = dpo_dataset.filter(
    lambda r: r["status"] != "tie" and r["chosen_score"] >= 8 and not r["in_gsm8k_train"]
)
dpo_dataset = dpo_dataset.map(format_prompt, remove_columns=dpo_dataset.column_names)
dpo_dataset

**(+ 보충)** 10절에서 "학습에 쓰지 않은 쌍"으로 선호 마진을 재려고 50쌍을 떼어 둡니다. 학습 데이터는 책보다 50개 적지만, 200스텝 × 8개 = 1,600개만 쓰므로 결과에는 사실상 영향이 없습니다.

In [ ]:
split = dpo_dataset.train_test_split(test_size=50, seed=42)
dpo_dataset, dpo_held_out = split["train"], split["test"]
print(dpo_held_out[0]["prompt"][:300])

SFT 어댑터를 4비트 베이스 모델에 불러와 병합하고, 그 위에 **새 LoRA**를 붙여 DPO를 학습합니다(4비트 병합이라 반올림 경고가 나오는 것은 정상).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16", bnb_4bit_use_double_quant=True,
)
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora", low_cpu_mem_usage=True, device_map="auto", quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = "<unk>"
tokenizer.padding_side = "left"

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32, lora_dropout=0.1, r=64, bias="none", task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj'],
)
model = prepare_model_for_kbit_training(merged_model)
model = get_peft_model(model, peft_config)

> **책과 다른 점.** ① `beta`, `max_length`가 `DPOTrainer` 인자에서 `DPOConfig`로 옮겨졌습니다. ② `max_prompt_length`는 현재 TRL의 `DPOConfig`에 없어서 뺐습니다. 프롬프트+응답 전체를 `max_length=512`로 자릅니다. ③ `warmup_ratio=0.1`은 transformers 5에서 `warmup_steps=0.1`(1 미만 실수는 비율로 해석)로 바뀌었습니다. ④ SFT 때와 같은 이유로 `peft_config`는 넘기지 않습니다. 참조 모델을 따로 주지 않으면 TRL은 **어댑터를 끈 상태의 모델**(= SFT 모델)을 참조 모델로 씁니다.

- `beta=0.1`: 참조 모델에서 얼마나 멀어지도록 허용할지. 작을수록 더 많이 바뀝니다.
- `learning_rate=1e-5`: SFT(2e-4)보다 훨씬 작게 둡니다.

In [ ]:
from trl import DPOConfig, DPOTrainer

training_arguments = DPOConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_steps=DPO_STEPS,            # 책: 200
    logging_steps=10 if not QUICK else 2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    warmup_steps=0.1,               # 책: warmup_ratio=0.1
    beta=0.1,                       # 책: DPOTrainer 인자
    max_length=512,                 # 책: DPOTrainer 인자 (max_prompt_length=512는 현재 없음)
    report_to="none",
)
dpo_trainer = DPOTrainer(
    model,
    args=training_arguments,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,     # 책: tokenizer=tokenizer
)
dpo_trainer.train()
dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

**(+ 보충) DPO 로그 읽기.** TRL은 학습 중 `rewards/chosen`, `rewards/rejected`(참조 모델 대비 로그 확률 변화 × beta)와 그 차이 `rewards/margins`, 그리고 chosen이 rejected보다 높게 매겨진 비율 `rewards/accuracies`를 기록합니다. 마진과 정확도가 올라가면 DPO가 의도대로 작동하는 것입니다. 손실은 log(2) ≈ 0.693에서 시작합니다(처음엔 학습 모델 = 참조 모델).

In [ ]:
dlog = pd.DataFrame([h for h in dpo_trainer.state.log_history if "rewards/margins" in h])
if len(dlog):
    cols = [c for c in ["loss", "rewards/margins", "rewards/accuracies"] if c in dlog]
    axes = dlog.plot(x="step", y=cols, subplots=True, marker="o", figsize=(7, 6), title="DPO training logs", legend=True)
    for a in axes: a.grid(alpha=.3)
else:
    print("기록된 DPO 로그가 없습니다.")

In [ ]:
del model, merged_model, dpo_trainer; free_gpu()

두 어댑터를 차례로 병합합니다: 베이스 + SFT 어댑터 → `sft_model`, 그 위에 DPO 어댑터 → `dpo_model`.

> 책은 아래 두 단계를 한 셀에서 합니다. `merge_and_unload()`는 가중치를 **제자리에서** 바꾸므로, DPO를 병합하고 나면 `sft_model`도 DPO 모델이 됩니다. 10절 비교에 SFT 상태가 필요해서 셀을 둘로 나누고, 그 사이에 SFT 측정을 넣었습니다.

In [ ]:
from peft import PeftModel

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora", low_cpu_mem_usage=True, device_map="auto", dtype=torch.float16,
)
sft_model = model.merge_and_unload()

**(+ 보충) DPO 병합 전에 SFT 모델의 선호 점수 기록.** 떼어 둔 50쌍에 대해 응답의 로그 확률 합 log p(응답 | 프롬프트)를 chosen·rejected 각각 계산합니다.

In [ ]:
def seq_logp(m, prompt, response, max_len=1024):
    p = tokenizer(prompt).input_ids                              # <s> 포함
    r = tokenizer(response, add_special_tokens=False).input_ids
    ids = torch.tensor([(p + r)[:max_len]], device=m.device)
    with torch.no_grad():
        logits = m(ids).logits.float()
    lp = torch.log_softmax(logits[:, :-1], -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return lp[0, len(p) - 1:].sum().item()                       # 응답 토큰 부분만 합산

def score_pairs(m):
    return pd.DataFrame([{"chosen": seq_logp(m, ex["prompt"], ex["chosen"]),
                          "rejected": seq_logp(m, ex["prompt"], ex["rejected"])} for ex in dpo_held_out])

sft_scores = score_pairs(sft_model)
sft_scores.describe().round(1)

In [ ]:
dpo_model = PeftModel.from_pretrained(sft_model, "TinyLlama-1.1B-dpo-qlora", device_map="auto")
dpo_model = dpo_model.merge_and_unload()

In [ ]:
from transformers import pipeline

prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
pipe = pipeline(task="text-generation", model=dpo_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

## 10. (+ 보충) 단계별 비교

**① 같은 질문, 세 모델.** 그리디 디코딩(`do_sample=False`)으로 같은 조건에서 생성한 답입니다. 베이스는 4비트, SFT·DPO는 16비트라 정밀도 차이도 조금 섞여 있습니다. 책 출력처럼 SFT와 DPO 답이 거의 같을 수 있습니다. DPO는 200스텝(1,600개)만 학습해 변화가 작고, 이 질문이 선호 데이터(OpenOrca 계열 추론·지시 문제)와 성격이 다르기 때문입니다.

In [ ]:
answers["2. SFT+DPO"] = ask(dpo_model)
for k, v in answers.items():
    print(f"===== {k}\n{v.strip()[:700]}\n")

**② 선호 마진.** 같은 50쌍을 DPO 모델로 다시 채점합니다.

- **마진** = log p(chosen) − log p(rejected). 클수록 사람이 고른 답을 더 그럴듯하다고 봅니다.
- **암묵적 보상 정확도:** DPO 모델이 SFT 모델(= 참조 모델) 대비 chosen의 로그 확률을 rejected보다 더 많이 올린 비율입니다. 학습 로그의 `rewards/accuracies`를 **학습에 쓰지 않은 데이터**에서 다시 잰 값입니다. 0.5보다 크면 DPO가 학습 데이터 밖에서도 선호 방향으로 움직였다는 뜻입니다. 마진의 절댓값은 응답 길이의 영향을 크게 받습니다(긴 답일수록 로그 확률 합이 작아짐). 같은 응답을 SFT와 DPO로 비교하는 암묵적 보상 정확도는 길이 효과가 상쇄되므로 이쪽을 주 지표로 보세요.

In [ ]:
dpo_scores = score_pairs(dpo_model)
margin = pd.DataFrame({"SFT": sft_scores.chosen - sft_scores.rejected,
                       "SFT+DPO": dpo_scores.chosen - dpo_scores.rejected})
implicit_acc = ((dpo_scores.chosen - sft_scores.chosen) > (dpo_scores.rejected - sft_scores.rejected)).mean()
print(f"평균 마진  SFT: {margin['SFT'].mean():.2f}  →  SFT+DPO: {margin['SFT+DPO'].mean():.2f}")
print(f"chosen 우세 비율  SFT: {(margin['SFT']>0).mean():.0%}  →  SFT+DPO: {(margin['SFT+DPO']>0).mean():.0%}")
print(f"암묵적 보상 정확도(held-out): {implicit_acc:.0%}")

In [ ]:
ax = margin.plot.box(figsize=(5, 3.5), title="log p(chosen) - log p(rejected) on 50 held-out pairs")
ax.axhline(0, color="gray", lw=.8); ax.grid(alpha=.3)

## 11. 정리

| 단계 | 데이터 | 학습 방법 | 결과물(이 노트북) |
|---|---|---|---|
| 사전학습 | 대량 텍스트 | 다음 토큰 예측 | TinyLlama 베이스 (내려받음) |
| SFT(지시 튜닝) | 질문-답 대화 3,000개 | QLoRA, 다음 토큰 예측 | `TinyLlama-1.1B-qlora` 어댑터 |
| 선호 튜닝 | prompt·chosen·rejected 약 5,900쌍 | QLoRA + DPO (참조 = SFT 모델) | `TinyLlama-1.1B-dpo-qlora` 어댑터 |

| 개념 | 한 줄 정리 |
|---|---|
| LoRA | 큰 행렬의 변화량을 작은 두 행렬 곱으로 학습. r=64에서 행렬당 약 6% |
| QLoRA | 얼린 원래 가중치는 4비트(nf4), 학습은 LoRA만. T4에서는 fp16 계산 |
| 보상 모델 + PPO | 채점 모델을 따로 학습해 강화학습. 강력하지만 복잡 |
| DPO | 참조 모델 대비 chosen↑ rejected↓를 직접 최적화. 보상 모델 없음 |

**자율주행·Agent로 이어지는 점(책 밖).** 주행 정책 모델의 학습도 같은 흐름을 따릅니다. 사람 운전 데이터를 따라 하는 모방학습(imitation learning)이 SFT에 해당합니다. 그다음 "더 안전하고 매끄러운 궤적"을 선호하도록 정렬하는 단계가 붙는데, 사람이나 규칙·시뮬레이터 점수로 궤적 쌍을 비교하면 DPO와 같은 선호 학습이 되고, 시뮬레이터(폐루프) 보상으로 학습하면 PPO 계열 강화학습이 됩니다. Agent 쪽에서는 도구 호출 형식을 지키게 하는 SFT, 더 나은 행동 순서를 고르게 하는 선호 튜닝이 같은 도구(TRL, PEFT)로 이뤄집니다. 7절의 결론("용도에 맞게 직접 평가하라")은 주행에서 오픈루프 지표와 폐루프 주행 점수가 다르게 나오는 문제와도 닿아 있습니다.

**끝나면** Studio에서 space를 **Stop** 하세요. 어댑터 폴더 2개(각 약 200MB)와 `./results`는 EBS에 남습니다.